In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/sepsis.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:group": "string",
        "case:age": "float32",
        "Leucocytes": "float32",
        "CRP": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,A,2014-10-22 11:15:41,0.0,0.0,85.0,ER Registration,complete,A,0
1,A,2014-10-22 11:27:00,0.0,9.6,85.0,Leucocytes,complete,B,679
2,A,2014-10-22 11:27:00,21.0,0.0,85.0,CRP,complete,B,0
3,A,2014-10-22 11:27:00,0.0,0.0,85.0,LacticAcid,complete,B,0
4,A,2014-10-22 11:33:37,0.0,0.0,85.0,ER Triage,complete,C,397
5,A,2014-10-22 11:34:00,0.0,0.0,85.0,ER Sepsis Triage,complete,A,23
6,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Liquid,complete,A,8987
7,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Antibiotics,complete,A,0
8,A,2014-10-22 14:13:19,0.0,0.0,85.0,Admission NC,complete,D,572
9,A,2014-10-24 09:00:00,109.0,0.0,85.0,CRP,complete,B,154001


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['CRP', 'Leucocytes', 'case:age', 'concept:name', 'lifecycle:transition', 'org:group', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
org:group                      categorical    event    yes    ['A', 'B', 'C', ...]                     N/A        data_derived        
case:age                       continuous     case     yes    [40.00, 90.00]                           10.0000    quantile_derived    
time_delta                     continuous     event    yes    [0.00, 38748.80]                         139.0000   quantile_derived    
Leucocytes                    

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,f_693,0,True,137.471281,12.525658,66.334927,Leucocytes,complete,O,36596.287463
1,f_693,1,True,49.415934,10.172167,66.334927,Admission IC,complete,W,22834.983431
2,f_693,2,True,188.040789,12.471686,66.334927,IV Antibiotics,complete,P,20261.712016
3,f_693,3,True,24.815061,7.751245,66.334927,CRP,complete,J,2767.638966
4,f_693,4,True,113.689623,11.711008,66.334927,LacticAcid,complete,H,5786.974602


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/sepsis-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0001 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 474.910128 seconds
Time taken for scenario model (validation): 0.458964 seconds
Val loss: {'loss': 0.0017345715059908049, 'accuracy': 0.9997551220308546, 'f1_macro': 0.9996630507862034, 'f1_weighted': 0.9997551076537083}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()